# Penalties analysis (roland_nb)

First-pass analysis for the solver penalty cap redesign. This is the working notebook;
other team members keep their own and we consolidate later.

How to read this:
- Data covers 7 chains (polygon, bnb, ethereum, arbitrum, base, gnosis, avalanche), 2026-01-06 to
  2026-06-21, one row per (auction_id, order_uid) winning settlement attempt (~1.4M fill-or-kill
  in-market rows). Magnitudes are illustrative; the method is the point.
- Analysis frame is fill-or-kill, in-market orders only (`~partially_fillable & ~is_out_of_market`).
- Native-token amounts are not comparable across chains, so we lead with cap-relative ratios, USD,
  and bps of order size. We use medians and ECDFs, not means, because the distributions are skewed.
- Only 3 chains means 3 cap levels, so cross-chain numbers are descriptive, not causal. Where we
  need a cleaner read we compare a solver against itself across the chains it operates on.

## 0. Setup and shared definitions

- load the three chains, parse the wei-string columns, apply the fok in-market filter, and
  define once the revert flag, the P&L formula, a per-chain native-USD price, and the outlier gate.
- every section below reuses these, so the definitions live in one place.

In [ ]:
import glob, os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
try: get_ipython().run_line_magic("matplotlib", "inline")
except Exception: pass

# Charts render as static PNG by default (always show, no JS needed). For interactive plotly
# (zoom, hover, legend toggle), set INTERACTIVE = True and re-run, or per cell call
# fig.show(renderer="plotly_mimetype+notebook").
INTERACTIVE = True
pio.renderers.default = "plotly_mimetype+notebook" if INTERACTIVE else "png"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 90)

WEI = 1e18
DATA_DIR = Path("../data")
# Chains are discovered from the data files; everything downstream iterates CHAIN_ORDER, so the
# notebook adapts to whatever chains are present. Known tokens/colors/block-times below, with
# fallbacks for any new chain.
_found = sorted({os.path.basename(p).split("_")[0] for p in glob.glob(str(DATA_DIR / "*.csv"))})
_pref = ["polygon", "bnb", "ethereum", "arbitrum", "base", "gnosis", "avalanche"]
CHAIN_ORDER = [c for c in _pref if c in _found] + [c for c in _found if c not in _pref]
_token = {"polygon": "POL", "bnb": "BNB", "ethereum": "ETH", "arbitrum": "ETH",
          "base": "ETH", "gnosis": "XDAI", "avalanche": "AVAX"}
NATIVE_TOKEN = {c: _token.get(c, c.upper()) for c in CHAIN_ORDER}
# approximate block times (seconds) for latency-in-blocks normalization; caveat only
_blk = {"polygon": 2.0, "bnb": 0.75, "ethereum": 12.0, "arbitrum": 0.25,
        "base": 2.0, "gnosis": 5.0, "avalanche": 2.0}
BLOCK_SECONDS = {c: _blk.get(c, np.nan) for c in CHAIN_ORDER}
_brand = {"polygon": "#8247E5", "bnb": "#F0B90B", "ethereum": "#627EEA", "arbitrum": "#28A0F0",
          "base": "#0052FF", "gnosis": "#3e6957", "avalanche": "#E84142"}
_pal = px.colors.qualitative.Safe
COLOR = {c: _brand.get(c, _pal[i % len(_pal)]) for i, c in enumerate(CHAIN_ORDER)}
print("chains:", CHAIN_ORDER)

def wilson(k, n, z=1.96):
    # Wilson score interval for a proportion, no scipy needed
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    centre = (p + z*z/(2*n)) / d
    half = (z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))) / d
    return (p, centre - half, centre + half)

def winsorize(s, lo=0.01, hi=0.99):
    s = pd.to_numeric(s, errors="coerce")
    ql, qh = s.quantile(lo), s.quantile(hi)
    return s.clip(ql, qh)

def ecdf_fig(df, value, title, logx=False, frame_settled_only=False):
    fig = go.Figure()
    for ch in CHAIN_ORDER:
        v = pd.to_numeric(df.loc[df.chain == ch, value], errors="coerce").dropna()
        if logx: v = v[v > 0]
        v = np.sort(v.values)
        if len(v) == 0: continue
        y = np.arange(1, len(v)+1) / len(v)
        # downsample for plot weight
        if len(v) > 3000:
            idx = np.linspace(0, len(v)-1, 3000).astype(int); v = v[idx]; y = y[idx]
        fig.add_scatter(x=v, y=y, mode="lines", name=ch, line=dict(color=COLOR[ch]))
    fig.update_layout(title=title, yaxis_title="ECDF", xaxis_title=value)
    if logx: fig.update_xaxes(type="log")
    return fig
print("helpers ready")

In [ ]:
# Load, parse, derive. Numeric columns are stored as decimal strings of wei to keep precision.
NUM = ["volume_native","penalty_native","penalty_uncapped_native","reward_penalty_native",
       "reward_penalty_uncapped_native","reward_native","penalty_cap_native","reward_cap_upper_native",
       "order_size_usd","markout_usd","markout_relative","reference_score","observed_score",
       "slippage_native","slippage_usd","execution_cost_native","seconds_since_created",
       "seconds_to_settle","slippage_tolerance_bps","calculated_slippage_bps","block_deadline",
       "settlement_block"]
BOOL = ["settled","is_excluded_from_penalties","partially_fillable","is_out_of_market","smart_slippage"]

def to_bool(s):
    return (s.astype(str).str.strip().str.lower()
            .map({"true": True, "false": False, "1": True, "0": False}).fillna(False).astype(bool))

frames, prov = [], []
for path in sorted(glob.glob(str(DATA_DIR / "*.csv"))):
    chain = os.path.basename(path).split("_")[0]
    d = pd.read_csv(path, low_memory=False)
    d.insert(0, "chain", chain)
    for c in NUM:
        if c in d: d[c] = pd.to_numeric(d[c], errors="coerce")
    for c in BOOL:
        if c in d: d[c] = to_bool(d[c])
    frames.append(d)
    prov.append((os.path.basename(path), len(d), hashlib.md5(open(path,"rb").read()).hexdigest()[:8]))

raw = pd.concat(frames, ignore_index=True)
_nexcl = int(raw["is_excluded_from_penalties"].sum())
if _nexcl:
    print(f"note: {_nexcl} rows flagged is_excluded_from_penalties (protocol-side, e.g. driver issues); "
          "dropped from the analysis frame so they do not pollute penalty/revert stats.")
print("provenance (file, rows, md5):"); [print("  ", *p) for p in prov]
print("loaded", len(raw), "rows")

In [ ]:
# fok in-market frame + derived columns. Drop is_excluded_from_penalties (protocol-side auctions,
# e.g. driver issues) so they do not pollute the penalty / revert statistics.
df = raw[(~raw.partially_fillable) & (~raw.is_out_of_market) & (~raw.is_excluded_from_penalties)].copy()
df["is_revert"] = ~df["settled"]

# native -> token
for c in ["volume_native","penalty_native","penalty_uncapped_native","reward_native",
          "reward_penalty_native","penalty_cap_native","reward_cap_upper_native",
          "slippage_native","execution_cost_native"]:
    df[c.replace("_native","_tok")] = df[c] / WEI

# per-chain native token USD price, implied from settled rows (order_size_usd / volume_tok)
sett = df[df.settled & (df.volume_tok > 0) & df.order_size_usd.notna()]
NATIVE_USD = (sett.assign(px=sett.order_size_usd / sett.volume_tok)
              .groupby("chain")["px"].median())
print("implied native-token USD price per chain:"); print(NATIVE_USD.round(4))
df["native_usd"] = df.chain.map(NATIVE_USD)

# USD size proxy available on ALL rows (incl reverts), and cap in USD
df["size_usd"] = df.volume_tok * df.native_usd
df["penalty_usd"] = df.penalty_tok * df.native_usd
df["reward_usd"] = df.reward_tok * df.native_usd
df["penalty_cap_usd"] = df.penalty_cap_tok * df.native_usd
df["penalty_uncapped_usd"] = df.penalty_uncapped_tok * df.native_usd

# P&L: settled = reward_penalty + slippage - exec cost; revert = reward_penalty (= -penalty)
df["pnl_tok"] = (df.reward_penalty_tok
                 + df.slippage_tok.fillna(0.0) - df.execution_cost_tok.fillna(0.0))
df["pnl_usd"] = df.pnl_tok * df.native_usd

# cap binding + forgiven amount (reverts only meaningful)
df["cap_binds"] = df.penalty_uncapped_tok > df.penalty_cap_tok
df["forgiven_tok"] = (df.penalty_uncapped_tok - df.penalty_tok).clip(lower=0)

# penalty as bps of order size (reverts)
df["penalty_bps"] = np.where(df.size_usd > 0, df.penalty_usd / df.size_usd * 1e4, np.nan)
CAP_USD = (df.groupby("chain").penalty_cap_usd.first()).reindex(CHAIN_ORDER)
print("penalty cap in USD per chain:"); print(CAP_USD.round(2))
print("rows in fok frame:", len(df))

## 1. Data exploration

- coverage, revert balance, missingness, the ethereum native-unit outliers, and how often the
  cap binds, per chain.
- fixes the baselines and exposes the confounders (size mix, token and solver concentration)
  before any cross-chain claim.

In [ ]:
# Coverage and headline per-chain table
rows = []
for ch in CHAIN_ORDER:
    g = df[df.chain == ch]
    p, lo, hi = wilson(int(g.is_revert.sum()), len(g))
    rows.append(dict(
        chain=ch, token=NATIVE_TOKEN[ch], rows=len(g),
        orders=g.order_uid.nunique(), solvers=g.solver.nunique(),
        revert_rate=round(p,4), rr_lo=round(lo,4), rr_hi=round(hi,4),
        cap_tok=round(g.penalty_cap_tok.iloc[0],4), cap_usd=round(g.penalty_cap_usd.iloc[0],2),
        cap_binds_rate=round(g.loc[g.is_revert,"cap_binds"].mean(),4),
        med_size_usd=round(g.size_usd.median(),1),
        med_markout_rel=round(g.markout_relative.median(),5),
        med_secs_settle=round(g.seconds_to_settle.median(),1),
    ))
coverage = pd.DataFrame(rows)
display(coverage)

fig = go.Figure()
fig.add_bar(x=coverage.chain, y=coverage.revert_rate,
            error_y=dict(type="data", symmetric=False,
                         array=coverage.rr_hi-coverage.revert_rate,
                         arrayminus=coverage.revert_rate-coverage.rr_lo),
            marker_color=[COLOR[c] for c in coverage.chain])
fig.update_layout(title="Revert rate by chain (fok in-market, Wilson 95% CI)", yaxis_tickformat=".0%")
fig.show()

In [ ]:
# Missingness: settled-only columns should be null exactly on reverts
sett_only = ["order_size_usd","markout_usd","markout_relative","seconds_to_settle",
             "slippage_usd","execution_cost_native"]
miss = (df.assign(grp=np.where(df.settled,"settled","revert"))
        .groupby("grp")[sett_only].apply(lambda x: x.notna().mean()).round(3))
print("share non-null by settled/revert (settled-only columns):"); display(miss)

# Ethereum native-unit outlier diagnostic
diag = (df.groupby("chain")
        .agg(vol_p50=("volume_tok","median"), vol_p99=("volume_tok", lambda s: s.quantile(.99)),
             vol_max=("volume_tok","max"),
             unc_p99=("penalty_uncapped_tok", lambda s: s.quantile(.99)),
             unc_max=("penalty_uncapped_tok","max")).reindex(CHAIN_ORDER))
print("volume_tok / penalty_uncapped_tok tails (note ethereum max vs p99 gap):"); display(diag.round(2))

In [ ]:
# Distributions: ECDF overlays (dropdown over a few key quantities)
quants = [("size_usd","order size (USD)", True),
          ("markout_relative","markout (relative)", False),
          ("penalty_bps","penalty as bps of size (reverts)", False),
          ("seconds_to_settle","time to happy moo (s, settled)", True)]
fig = go.Figure(); buttons = []; traces_per = {}
for qi,(col,label,lx) in enumerate(quants):
    start = len(fig.data)
    sub = df if col != "penalty_bps" else df[df.is_revert]
    for ch in CHAIN_ORDER:
        v = pd.to_numeric(sub.loc[sub.chain==ch, col], errors="coerce").dropna()
        if lx: v = v[v>0]
        v = np.sort(v.values)
        if len(v)==0:
            fig.add_scatter(x=[], y=[], name=ch, visible=(qi==0)); continue
        y = np.arange(1,len(v)+1)/len(v)
        if len(v)>3000:
            idx=np.linspace(0,len(v)-1,3000).astype(int); v=v[idx]; y=y[idx]
        fig.add_scatter(x=v,y=y,mode="lines",name=ch,line=dict(color=COLOR[ch]),visible=(qi==0))
    traces_per[qi]=(start,len(fig.data))
n=len(fig.data)
for qi,(col,label,lx) in enumerate(quants):
    vis=[False]*n
    for t in range(*traces_per[qi]): vis[t]=True
    buttons.append(dict(label=label,method="update",
                        args=[{"visible":vis},{"xaxis":{"title":label,"type":"log" if lx else "linear"}}]))
fig.update_layout(title="Distributions by chain (ECDF)",
                  updatemenus=[dict(buttons=buttons,x=1.02,y=1.0)],
                  yaxis_title="ECDF", xaxis_title=quants[0][1],
                  xaxis_type="log")
fig.show()

In [ ]:
# Concentration: solver share and HHI per chain
def hhi(counts):
    s = counts/counts.sum(); return float((s*s).sum())
conc = []
for ch in CHAIN_ORDER:
    g = df[df.chain==ch]
    sc = g.solver_name.value_counts()
    conc.append(dict(chain=ch, n_solvers=g.solver.nunique(), solver_hhi=round(hhi(sc),3),
                     top_solver=sc.index[0] if len(sc) else None,
                     top_solver_share=round(sc.iloc[0]/len(g),3) if len(sc) else np.nan,
                     token_pairs=g.groupby(["sell_token","buy_token"]).ngroups))
display(pd.DataFrame(conc))

## 2. Revert rates across chains vs rewards and penalties

- per-chain revert rate next to realized rewards, realized penalties, and how often the cap binds.
- reverts are the event the cap targets; this frames whether penalties look like a deterrent or a cost solvers absorb.
- chains differ in flow and competition, so read cross-chain as description, not cause.

In [ ]:
t1 = []
for ch in CHAIN_ORDER:
    g = df[df.chain==ch]; rev = g[g.is_revert]
    p,lo,hi = wilson(int(g.is_revert.sum()), len(g))
    t1.append(dict(chain=ch, revert_rate=round(p,4),
                   med_reward_usd=round(g.loc[g.settled,"reward_usd"].median(),3),
                   med_penalty_usd=round(rev.penalty_usd.median(),3),
                   cap_usd=round(g.penalty_cap_usd.iloc[0],2),
                   cap_binds_rate=round(rev.cap_binds.mean(),3),
                   exp_penalty_per_attempt_usd=round(p*rev.penalty_usd.median(),4)))
t1 = pd.DataFrame(t1); display(t1)

# revert rate vs cap, bubble = expected penalty per attempt (one point per chain)
fig = px.scatter(t1, x="cap_usd", y="revert_rate", text="chain", size="exp_penalty_per_attempt_usd",
                 color="chain", color_discrete_map=COLOR, log_x=True, size_max=40,
                 title="Revert rate vs penalty cap (bubble = E[penalty] per attempt, USD)")
fig.update_traces(textposition="top center")
fig.update_yaxes(tickformat=".0%"); fig.update_xaxes(title="penalty cap (USD, log)"); fig.show()

# realized reward (settled) vs penalty (revert) per chain, chains ordered by cap
o = t1.sort_values("cap_usd")
fig = go.Figure()
fig.add_bar(y=o.chain, x=o.med_reward_usd, name="median reward (settled)", orientation="h", marker_color="seagreen")
fig.add_bar(y=o.chain, x=-o.med_penalty_usd, name="median penalty (revert)", orientation="h", marker_color="indianred")
fig.update_layout(barmode="relative", title="Realized reward vs penalty per chain (chains ordered by cap)",
                  xaxis_title="USD (penalty shown negative)"); fig.show()

## 3. Revert rates by order size

- revert rate across order-size buckets, within each chain.
- size is the prime confounder and the lever for a variable-rate cap; we also check whether fixed penalties land hardest on small orders (penalty in bps of size).
- size uses `size_usd` (present on reverts); `order_size_usd` is settled-only.

In [ ]:
# USD size bands (comparable across chains)
bands = [0,100,1_000,10_000,100_000,np.inf]
labels = ["<$100","$100-1k","$1k-10k","$10k-100k",">$100k"]
df["size_band"] = pd.cut(df.size_usd, bands, labels=labels)
rs = []
for ch in CHAIN_ORDER:
    for b in labels:
        g = df[(df.chain==ch)&(df.size_band==b)]
        if len(g)<30: continue
        p,lo,hi = wilson(int(g.is_revert.sum()), len(g))
        rs.append(dict(chain=ch, band=b, n=len(g), revert_rate=p, lo=lo, hi=hi,
                       med_penalty_bps=g.loc[g.is_revert,"penalty_bps"].median()))
rs = pd.DataFrame(rs)
fig = px.line(rs, x="band", y="revert_rate", color="chain", markers=True,
              color_discrete_map=COLOR, category_orders={"band":labels},
              title="Revert rate by order-size band")
fig.update_yaxes(tickformat=".0%"); fig.show()

fig2 = px.line(rs, x="band", y="med_penalty_bps", color="chain", markers=True,
               color_discrete_map=COLOR, category_orders={"band":labels},
               title="Median penalty as bps of order size, by size band (reverts)")
fig2.show()

In [ ]:
# Static scatters (matplotlib): revert rate vs size (binned) and penalty vs size (raw, log-log
# with the cap line). Reverts are 0/1 so revert-rate is binned; penalty is raw to show cap saturation.
allf = df[df.size_usd.notna() & (df.size_usd>0)].copy()
allf["sb"] = pd.cut(allf.size_usd, np.logspace(0,6,16))
figm, ax = plt.subplots(1,2, figsize=(13,4.5))
for ch in CHAIN_ORDER:
    g = allf[allf.chain==ch].groupby("sb", observed=True).agg(n=("is_revert","size"), rr=("is_revert","mean"))
    g = g[g.n>=50]
    if len(g): ax[0].plot([iv.mid for iv in g.index], g.rr, "o-", ms=4, color=COLOR[ch], label=ch, alpha=.8)
ax[0].set_xscale("log"); ax[0].set(xlabel="order size (USD)", ylabel="revert rate",
                                    title="Revert rate vs size (binned)"); ax[0].legend(fontsize=7)
revp = df[df.is_revert & (df.size_usd>0)]
for ch in ["polygon","ethereum","bnb"]:
    g = revp[revp.chain==ch]
    if len(g)>4000: g = g.sample(4000, random_state=0)
    ax[1].scatter(g.size_usd, g.penalty_usd, s=3, alpha=.15, color=COLOR[ch], label=ch)
    ax[1].axhline(CAP_USD[ch], color=COLOR[ch], ls="--", lw=1)
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set(xlabel="order size (USD)", ylabel="realized penalty (USD)",
          title="Penalty vs size (reverts; dashed = cap)"); ax[1].legend(fontsize=7)
figm.tight_layout(); plt.show()

## 4. Does a higher penalty cap go with lower rewards and lower solver P&L?

- per-chain rewards and P&L vs cap, plus the cleaner within-solver comparison (solvers active on more than one chain, compared against themselves).
- the core economic question, and P&L is the discriminator for the ambiguous null (flat markout + negative P&L means solvers eat the cap; reward and penalty caps are asymmetric).

In [ ]:
# P&L formula reminder: settled = reward_penalty + slippage - exec_cost ; revert = -penalty.
# Caveat: pnl charges the full penalty, but CIP-85 redistributes much of it back via the consistency
# budget, so net P&L overstates loss (more on chains with active redistribution). Use as directional.
t = df.groupby("chain").agg(
        med_pnl_usd=("pnl_usd","median"),
        neg_pnl_share=("pnl_usd", lambda s:(s<0).mean()),
        med_reward_usd=("reward_usd", lambda s: s[df.loc[s.index,"settled"]].median()),
    ).reindex(CHAIN_ORDER)
t["cap_usd"] = CAP_USD; t = t.reset_index()
display(t.round(4))

# Cross-chain: does a higher cap line up with more negative-P&L attempts? One point per chain.
fig = px.scatter(t, x="cap_usd", y="neg_pnl_share", text="chain", color="chain",
                 color_discrete_map=COLOR, log_x=True,
                 title="Share of attempts with negative P&L vs penalty cap (one point per chain)")
fig.update_traces(textposition="top center", marker=dict(size=12))
fig.update_yaxes(tickformat=".0%"); fig.update_xaxes(title="penalty cap (USD, log)"); fig.show()

# within-solver control (solvers on >=2 chains, >=100 rows/chain) reported as a number, not a noisy
# 6-line chart. cap_rank / multi are reused by Section 5.
ps = (df.groupby(["solver","chain"]).agg(n=("pnl_usd","size"), med_pnl=("pnl_usd","median")).reset_index())
ps = ps[ps.n>=100]
multi = ps.groupby("solver").chain.nunique(); multi = multi[multi>=2].index
cap_rank = {c:r for r,c in enumerate(CAP_USD.sort_values().index)}
pm = ps[ps.solver.isin(multi)].copy(); pm["cap_rank"] = pm.chain.map(cap_rank)
diffs = []
for s,g in pm.groupby("solver"):
    g = g.sort_values("cap_rank"); diffs.append(g.iloc[-1].med_pnl - g.iloc[0].med_pnl)
diffs = pd.Series(diffs, dtype=float)
print(f"within-solver control: {len(multi)} solvers on >=2 chains (>=100 rows/chain). Median P&L diff "
      f"(high-cap minus low-cap chain) = {diffs.median():.4f} USD/attempt; "
      f"{(diffs<0).mean():.0%} are lower on their higher-cap chain. n small, directional only.")

## 5. Is a higher cap priced into worse bids (worse markout)?

- markout per chain, and within-solver markout on the higher-cap vs lower-cap chain.
- tests whether cap risk is passed to users as worse execution.
- markout is settled-only, so strategic reverting of bad trades can make survivors look better (survivorship); read jointly with Section 2.

In [ ]:
# markout_relative has data-artifact extremes (max ~7e6 on arbitrum); clip to |x| p99 to read the
# ECDF, and show markout_usd as a boxplot (its tail is far tamer and is the more trustworthy field).
m = df.loc[df.settled, ["chain","markout_relative"]].dropna()
lim = float(np.nanpercentile(m.markout_relative.abs(), 99))
mc = m.assign(markout_clip=m.markout_relative.clip(-lim, lim))
fig = ecdf_fig(mc, "markout_clip", f"Markout (relative) by chain, settled (clipped to +/-{lim:.3f} = |x| p99)")
fig.add_vline(x=0, line_dash="dash", line_color="gray"); fig.show()

figm, ax = plt.subplots(figsize=(9,4)); s2 = df[df.settled]
data = [pd.to_numeric(s2.loc[s2.chain==ch,"markout_usd"], errors="coerce").dropna().values for ch in CHAIN_ORDER]
bp = ax.boxplot(data, tick_labels=CHAIN_ORDER, showfliers=False, patch_artist=True, whis=(10,90))
for patch,ch in zip(bp["boxes"], CHAIN_ORDER): patch.set_facecolor(COLOR[ch]); patch.set_alpha(.7)
for med in bp["medians"]: med.set_color("black")
ax.axhline(0, color="gray", lw=.6, ls=":"); ax.tick_params(axis="x", rotation=45)
ax.set(ylabel="markout (USD)", title="Markout (USD) by chain (box=IQR, whiskers 10/90)")
figm.tight_layout(); plt.show()

# within-solver markout (same multi-chain solver set)
psm = (df[df.settled].groupby(["solver","chain"]).markout_relative.median().reset_index())
psm = psm[psm.solver.isin(multi)].copy(); psm["cap_rank"]=psm.chain.map(cap_rank)
mp = []
for s,g in psm.groupby("solver"):
    g=g.sort_values("cap_rank")
    if g.chain.nunique()<2: continue
    mp.append(dict(solver=s, mk_low=g.iloc[0].markout_relative, mk_high=g.iloc[-1].markout_relative,
                   diff=g.iloc[-1].markout_relative-g.iloc[0].markout_relative))
mp=pd.DataFrame(mp)
if len(mp):
    print("median within-solver markout diff (high-cap minus low-cap):", round(mp["diff"].median(),6))
    print("share with worse (lower) markout on higher-cap chain:", round((mp["diff"]<0).mean(),3))
display(mp.round(6))

## 6. Three outcomes vs cap level: profit, markout, time to happy moo

- solver P&L, markout, and settle latency lined up against each chain's cap.
- the redesign needs the whole trade-off; hypothesis is higher caps go with worse markout and slower settlement.
- latency is shown in seconds and in blocks (chains have very different block times).

In [ ]:
o = (df[df.settled].groupby("chain")
     .agg(med_pnl_usd=("pnl_usd","median"),
          med_markout=("markout_relative","median"),
          med_secs=("seconds_to_settle","median")).reindex(CHAIN_ORDER))
o["med_blocks"] = o.med_secs / pd.Series(BLOCK_SECONDS).reindex(o.index)
o["cap_usd"] = CAP_USD
o = o.reset_index().sort_values("cap_usd")
display(o.round(4))

# Distributions as boxplots per chain (chains ordered low to high cap). The median table above keeps
# the cap relationship; the boxplots show the spread behind each median. Use the metric dropdown.
# Faceted static boxplots, one per metric, each with its own robust y-range (chains ordered by cap).
# The previous single go.Box dropdown collapsed all chains onto x=0 and shared one y-range.
chains_by_cap = CAP_USD.sort_values().index.tolist()
sett = df[df.settled]
metrics = [("pnl_usd","net P&L (USD)"), ("markout_relative","markout (relative)"),
           ("seconds_to_settle","time to happy moo (s)")]
figm, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax,(col,label) in zip(axes, metrics):
    data = [pd.to_numeric(sett.loc[sett.chain==ch,col], errors="coerce").dropna().values for ch in chains_by_cap]
    q = pd.to_numeric(sett[col], errors="coerce")
    bp = ax.boxplot(data, tick_labels=chains_by_cap, showfliers=False, patch_artist=True, whis=(10,90))
    for patch,ch in zip(bp["boxes"], chains_by_cap): patch.set_facecolor(COLOR[ch]); patch.set_alpha(.7)
    for med in bp["medians"]: med.set_color("black")
    ax.set_title(label); ax.axhline(0, color="gray", lw=.6, ls=":"); ax.tick_params(axis="x", rotation=45)
    if col=="pnl_usd":
        v = float(q.abs().quantile(.98)); ax.set_yscale("symlog", linthresh=max(v/50, 1e-3)); ax.set_ylim(-v, v)
    else:
        ax.set_ylim(q.quantile(.02), q.quantile(.98))
figm.suptitle("Outcome distributions by chain (ordered low->high cap; box=IQR, whiskers=10/90 pct)")
figm.tight_layout(); plt.show()

## 7. Distribution of capped vs uncapped penalties, per chain

- how often the cap binds and how much penalty it forgives.
- sizes the problem; if the cap rarely binds it is a non-event, if it clips heavily the redesign matters.
- uncapped penalty is the most ethereum-outlier-exposed field, so totals are winsorized and the raw is shown beside.

In [ ]:
rev = df[df.is_revert].copy()
t6 = []
for ch in CHAIN_ORDER:
    g = rev[rev.chain==ch]
    forg_raw = g.forgiven_tok.sum()
    forg_w = winsorize(g.forgiven_tok).sum()
    t6.append(dict(chain=ch, reverts=len(g), cap_binds_rate=round(g.cap_binds.mean(),3),
                   total_penalty_tok=round(g.penalty_tok.sum(),2),
                   forgiven_tok_winsor=round(forg_w,2), forgiven_tok_raw=round(forg_raw,2)))
display(pd.DataFrame(t6))

# ECDF of uncapped penalty relative to cap (log x), vertical line at 1 = cap
rev["unc_over_cap"] = rev.penalty_uncapped_tok / rev.penalty_cap_tok
fig = ecdf_fig(rev, "unc_over_cap", "Uncapped penalty / cap, by chain (reverts)", logx=True)
fig.add_vline(x=1.0, line_dash="dash", line_color="gray")
fig.show()

## 8. Counterfactual: variable-rate penalty (% of order size)

- recompute penalties if the cap were a flat rate on order size, holding observed reverts fixed,
  and find the revenue-neutral rate per chain under the capped formula `min(uncapped, r * size)`.
- the leading redesign candidate; we show who pays more or less and how penalty mass shifts by size.
- the revenue-neutral rate is found by bisection on the capped formula, NOT by `realized / total_size`.
  The naive linear rate is wrong here: many reverts have an uncapped penalty below `r * size`, so they
  never reach the rate, and the linear estimate understates the rate needed (badly on bnb, where the
  cap rarely binds). We report both so the gap is visible.
- assumes solvers do not change behavior (so it understates large-order relief); size uses the USD
  proxy with per-chain p99.9 outlier gating (ethereum native-unit artifacts).

In [ ]:
# size gate: per-order reverts, drop USD-size scaling artifacts above per-chain p99.9
gate = df[df.is_revert & df.size_usd.notna() & (df.size_usd > 0)].copy()
gate = gate[gate.groupby("chain").size_usd.transform(lambda s: s < s.quantile(0.999))]

def rn_bps_capped(unc, vol, target, lo=0.0, hi=1000.0, iters=60):
    # revenue-neutral bps solving sum(min(unc, r/1e4 * vol)) == target, by bisection.
    # monotone in r -> bisection is exact. Correct revenue-neutral rate under a capped penalty.
    for _ in range(iters):
        mid = (lo + hi) / 2
        tot = np.minimum(unc, mid / 1e4 * vol).sum()
        lo, hi = (mid, hi) if tot < target else (lo, mid)
    return (lo + hi) / 2

grid = [1, 2, 5, 10, 25, 50]
rows_sweep, rows_rn = [], []
for ch in CHAIN_ORDER:
    g = gate[gate.chain == ch]
    unc, vol = g.penalty_uncapped_usd.values, g.size_usd.values
    realized = g.penalty_usd.sum()
    for r in grid:
        tot = np.minimum(unc, r / 1e4 * vol).sum()
        rows_sweep.append(dict(chain=ch, rate_bps=r, capped_total_usd=tot,
                               realized_usd=realized, ratio=tot / realized if realized else np.nan))
    r_bisect = rn_bps_capped(unc, vol, realized)
    r_linear = realized / vol.sum() * 1e4 if vol.sum() else np.nan   # naive comparator (wrong here)
    rows_rn.append(dict(chain=ch, reverts=len(g), realized_usd=round(realized, 1),
                        rn_bps_capped=round(r_bisect, 2), rn_bps_linear_naive=round(r_linear, 2)))
sweep = pd.DataFrame(rows_sweep); rn = pd.DataFrame(rows_rn)
print("revenue-neutral penalty rate per chain (capped-formula bisection is the correct column):")
display(rn)

# the ~3.6x bnb-vs-polygon gap means a single uniform rate is NOT revenue-neutral across chains
rn_map = rn.set_index("chain").rn_bps_capped.to_dict()
r_uni = rn_map["polygon"]
uni = []
for ch in CHAIN_ORDER:
    g = gate[gate.chain == ch]
    tot = np.minimum(g.penalty_uncapped_usd, r_uni / 1e4 * g.size_usd).sum()
    uni.append(dict(chain=ch, uniform_rate_bps=round(r_uni, 2), collected_usd=round(tot, 0),
                    realized_usd=round(g.penalty_usd.sum(), 0),
                    ratio=round(tot / g.penalty_usd.sum(), 2) if g.penalty_usd.sum() else np.nan))
print(f"uniform-rate check: a rate revenue-neutral on polygon ({r_uni:.1f} bps) over/under-collects elsewhere:")
display(pd.DataFrame(uni))

fig = px.line(sweep, x="rate_bps", y="ratio", color="chain", markers=True, color_discrete_map=COLOR,
              title="Variable-rate total vs realized fixed-cap (1.0 = revenue neutral)")
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_xaxes(title="penalty rate (bps of order size)"); fig.show()

In [ ]:
# Alternative view (static): the current FIXED cap is regressive in bps of size. Effective penalty
# as bps of size by size decile, per chain. A flat variable rate would be a horizontal line here, so
# the slope IS the redesign argument (small orders pay a far higher effective rate than large ones).
gate2 = df[df.is_revert & (df.size_usd>0)].copy()
gate2 = gate2[gate2.groupby("chain").size_usd.transform(lambda s: s < s.quantile(.999))]
figm, ax = plt.subplots(figsize=(8.5,4.5))
for ch in CHAIN_ORDER:
    g = gate2[gate2.chain==ch].copy()
    if len(g) < 200: continue
    g["dec"] = pd.qcut(g.size_usd, 10, labels=False, duplicates="drop")
    eff = g.groupby("dec").apply(lambda d: d.penalty_usd.sum()/d.size_usd.sum()*1e4, include_groups=False)
    msz = g.groupby("dec").size_usd.median()
    ax.plot(msz.values, eff.values, "o-", ms=4, color=COLOR[ch], label=ch, alpha=.85)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set(xlabel="median order size in decile (USD)", ylabel="effective penalty (bps of size)",
       title="Current fixed cap is regressive: effective bps falls steeply with size")
ax.legend(fontsize=7); figm.tight_layout(); plt.show()

## 8b. Counterfactual: protocol-fee cap (data gap, with illustrative proxy)

- target design: penalty capped at the protocol's own fee on that settlement,
  `penalty = min(uncapped, m * protocolFee_i)`, where the reward cap is meant to be
  `reward_cap_upper_native = beta * protocolFee_i`.
- DATA GAP: in this extract `reward_cap_upper_native` is a single CONSTANT per chain (polygon 40,
  bnb 0.048, ethereum 0.012 tok), exactly 1.2-1.33x the penalty cap, and it binds the reward on
  <0.1% of settled rows. It is a fixed protocol PARAMETER, not an endogenous `beta * protocolFee_i`,
  so `protocolFee_i = reward_cap_upper / beta` is constant and useless as a per-order cap. There is
  no per-settlement protocol-fee column. The real protocol-fee cap is NOT computable from this data.
- to recover it we need a per-settlement `protocol_fee_native` column (fee actually charged on the
  winning solution), or surplus + the fee policy (bps-of-surplus and its cap) to reconstruct it.
- the cell below confirms the constancy, then runs a clearly-labelled ILLUSTRATIVE surplus-proxy fee
  only to make the large-order design point visible: a fee-sized cap covers a small fraction of the
  harm from a failed large order, so a pure protocol-fee cap is too lenient there (Archit's argument).

In [ ]:
# Step 1: is protocolFee_i recoverable from reward_cap_upper? (No.)
rec = []
for ch in CHAIN_ORDER:
    g = df[df.chain == ch]; rcu = g.reward_cap_upper_tok
    rcu_usd = g.reward_cap_upper_tok.iloc[0] * g.native_usd.iloc[0]
    rec.append(dict(chain=ch, reward_cap_upper_tok=round(rcu.iloc[0], 6), n_unique=rcu.nunique(),
                    reward_cap_upper_usd=round(rcu_usd, 2),
                    ratio_to_penalty_cap=round((g.reward_cap_upper_tok / g.penalty_cap_tok).iloc[0], 3),
                    share_settled_reward_at_cap=round(
                        np.isclose(g.loc[g.settled, "reward_tok"], rcu.iloc[0], rtol=0.02).mean(), 4)))
print("reward_cap_upper is constant per chain (n_unique=1) => NOT endogenous => protocol fee NOT recoverable:")
display(pd.DataFrame(rec))

# Step 2: ILLUSTRATIVE surplus-proxy fee, to show the large-order leniency point ONLY.
# ASSUMPTIONS (state clearly, not measured): gross surplus ~ SURPLUS_BPS of size; protocol takes
# PROTO_SHARE of it. protocolFee_proxy = size_usd * SURPLUS_BPS/1e4 * PROTO_SHARE.
SURPLUS_BPS, PROTO_SHARE = 10.0, 0.5
g = gate.copy()
g["protofee_proxy_usd"] = g.size_usd * SURPLUS_BPS / 1e4 * PROTO_SHARE
rows = []
for m in [1, 2, 5]:
    for ch in CHAIN_ORDER:
        d = g[g.chain == ch]
        pen = np.minimum(d.penalty_uncapped_usd, m * d.protofee_proxy_usd).sum()
        rows.append(dict(multiplier_m=m, chain=ch, feecap_total_usd=round(pen, 0),
                         fixed_total_usd=round(d.penalty_usd.sum(), 0),
                         ratio_vs_fixed=round(pen / d.penalty_usd.sum(), 2) if d.penalty_usd.sum() else np.nan))
print(f"ILLUSTRATIVE protocol-fee cap (surplus={SURPLUS_BPS:.0f} bps, share={PROTO_SHARE:.0%}); ASSUMED, not measured:")
display(pd.DataFrame(rows))

# large-order leniency: top size decile of reverts, uncapped harm vs the fee-proxy cap
big_rows = []
for ch in CHAIN_ORDER:
    d = g[g.chain == ch]; thr = d.size_usd.quantile(0.9); big = d[d.size_usd >= thr]
    big_rows.append(dict(chain=ch, n=len(big),
                         med_uncapped_usd=round(big.penalty_uncapped_usd.median(), 2),
                         med_protofee_proxy_usd=round(big.protofee_proxy_usd.median(), 2),
                         share_fee_covers_harm=round((big.protofee_proxy_usd >= big.penalty_uncapped_usd).mean(), 2)))
print("large-order (top size decile of reverts): a fee-sized cap covers only a fraction of the harm")
display(pd.DataFrame(big_rows))

## 8c. Who gains and who loses: variable-rate vs current fixed cap

- per size band and per solver, in penalty-budget-share terms, who pays more vs less if the cap
  becomes a variable rate at each chain's revenue-neutral bps (revenue held constant by construction).
- the redesign's distributional question: a variable rate is budget-neutral overall but moves mass
  from small orders onto large orders, and from small-order solvers onto large-order solvers.

In [ ]:
rn_map = rn.set_index("chain").rn_bps_capped.to_dict()
g = gate.copy()
g["pen_fixed"] = g.penalty_usd
g["pen_var"] = np.minimum(g.penalty_uncapped_usd, g.chain.map(rn_map) / 1e4 * g.size_usd)
bands = [0, 100, 1_000, 10_000, 100_000, np.inf]
labels = ["<$100", "$100-1k", "$1k-10k", "$10k-100k", ">$100k"]
g["band"] = pd.cut(g.size_usd, bands, labels=labels)

# budget share by size band (per chain)
share_rows = []
for ch in CHAIN_ORDER:
    d = g[g.chain == ch]; tf, tv = d.pen_fixed.sum(), d.pen_var.sum()
    bb = d.groupby("band", observed=True).agg(fix=("pen_fixed", "sum"), var=("pen_var", "sum"))
    for b, r in bb.iterrows():
        share_rows.append(dict(chain=ch, band=b,
                               fixed_share=r.fix / tf if tf else np.nan,
                               var_share=r["var"] / tv if tv else np.nan,
                               delta_share=(r["var"] / tv if tv else 0) - (r.fix / tf if tf else 0)))
band_share = pd.DataFrame(share_rows)
print("penalty budget share by size band, fixed cap vs variable @ rev-neutral (delta>0 = pays more):")
display(band_share.round(3))

fig = px.bar(band_share, x="band", y="delta_share", color="chain", barmode="group",
             color_discrete_map=COLOR, category_orders={"band": labels},
             title="Penalty budget-share shift by size band (variable @ rev-neutral minus fixed cap)")
fig.add_hline(y=0, line_color="gray"); fig.update_yaxes(tickformat=".0%"); fig.show()

# budget share by solver (all chains pooled), min 20 gated reverts
gs = g.groupby("solver_name").agg(fix=("pen_fixed", "sum"), var=("pen_var", "sum"), n=("pen_fixed", "size"))
gs = gs[gs.n >= 20]
tf, tv = g.pen_fixed.sum(), g.pen_var.sum()
gs["fixed_share"] = gs.fix / tf; gs["var_share"] = gs["var"] / tv
gs["delta_share"] = gs.var_share - gs.fixed_share
gs = gs.sort_values("delta_share")
print("solvers paying LESS under variable rate (small-order solvers):")
display(gs[["n", "fixed_share", "var_share", "delta_share"]].head(5).round(4))
print("solvers paying MORE under variable rate (large-order solvers):")
display(gs[["n", "fixed_share", "var_share", "delta_share"]].tail(5).round(4))

## 9. Counterfactual: non-economic penalty (sit out N auctions)

- approximate suspending a reverting solver for N auctions, with exponential backoff on repeat
  reverts; report what it costs the solver and what it risks for users.
- a non-economic alternative to a monetary cap. Felix's framing: start at 1 auction, escalate on
  repeat offenses. This does NOT change overbidding incentives, it only rations participation.
- time unit is the auction, not the block. block_deadline is 1:1 and monotone with auction_id within
  a chain, but the block gap between consecutive auctions is ~2 on ethereum vs ~40-50 on
  polygon/bnb, so a fixed block ban is not comparable across chains.
- we only see winning attempts, so removing a solver's settled win counts as forgone reward and as
  coverage-at-risk unless some other solver settled that same order. On this sample every order
  settles once, so coverage-at-risk equals removed settled wins, an upper bound: in reality the
  runner-up bidder settles most of them. forgone reward and the settled-rate delta are upper bounds
  for the same reason.

In [ ]:
# Sit-out with exponential backoff. Unit = AUCTION (order by block_deadline; ban the solver from
# its next N winning attempts). ban = min(cap, base*factor**(offense-1)); offense resets after a
# clean streak. The TRIGGERING revert is not removed (it precedes the ban, penalty still incurred).
# No avoided-penalty accumulation: a banned solver's prevented revert is not a cost it dodged.
ORDER_SETTLES = (df[df.settled].sort_values("block_deadline")
                 .groupby("order_uid").agg(bd=("block_deadline", list), sv=("solver", list)))

def sim_sitout(g, base=1, factor=2, cap=64, reset_clean=10):
    g = g.sort_values("block_deadline"); state = {}
    removed = removed_settled = coverage_at_risk = 0
    forgone_reward = triggering_penalty = 0.0
    for r in g.itertuples():
        s = r.solver; st = state.get(s)
        if st and st["ban"] > 0:                      # suspended: drop this attempt
            st["ban"] -= 1; removed += 1
            if r.settled:
                removed_settled += 1
                forgone_reward += 0.0 if np.isnan(r.reward_usd) else r.reward_usd
                covered = False
                if r.order_uid in ORDER_SETTLES.index:
                    rec = ORDER_SETTLES.loc[r.order_uid]
                    for bd, sv in zip(rec["bd"], rec["sv"]):
                        if bd > r.block_deadline and sv != s: covered = True; break
                if not covered: coverage_at_risk += 1
            continue
        if r.is_revert:                               # active revert: trigger / escalate ban
            off = (st["offense"] + 1) if st else 1
            state[s] = {"ban": int(min(cap, base * factor**(off-1))), "offense": off, "clean": 0}
            triggering_penalty += 0.0 if np.isnan(r.penalty_usd) else r.penalty_usd
        elif st:                                      # active clean attempt: maybe reset offense
            st["clean"] += 1
            if st["clean"] >= reset_clean: st["offense"] = 0; st["clean"] = 0
    return dict(removed=removed, removed_settled=removed_settled, forgone_reward_usd=forgone_reward,
                triggering_penalty_usd=triggering_penalty, coverage_at_risk=coverage_at_risk)

SCHEDULES = {
    "1 auction (flat)":   dict(base=1, factor=1, cap=1,  reset_clean=10),   # Felix's starting point
    "1,2,4,... cap 64":   dict(base=1, factor=2, cap=64, reset_clean=10),   # exponential backoff
    "2,4,8,... cap 64":   dict(base=2, factor=2, cap=64, reset_clean=10),   # steeper backoff
    "5 auctions (flat)":  dict(base=5, factor=1, cap=5,  reset_clean=10),   # fixed reference
}
rows = []
for name, kw in SCHEDULES.items():
    for ch in CHAIN_ORDER:
        g = df[df.chain == ch]; n_att = len(g); n_set = int(g.settled.sum())
        d = sim_sitout(g, **kw); new_set = n_set - d["removed_settled"]
        rows.append(dict(schedule=name, chain=ch, removed=d["removed"],
            removed_settled=d["removed_settled"], forgone_reward_usd=round(d["forgone_reward_usd"], 2),
            coverage_at_risk=d["coverage_at_risk"], settled_rate=round(n_set/n_att, 4),
            settled_rate_cf=round(new_set/n_att, 4),
            settled_rate_delta_bps=round(1e4*(new_set-n_set)/n_att, 1)))
sit = pd.DataFrame(rows)
display(sit)
print("coverage-at-risk equals removed settled wins here (each order settles once): a strict upper")
print("bound, since the runner-up bidder would settle most of these. forgone reward and the")
print("settled-rate delta are upper bounds too. Sit-out does not change overbidding incentives.")

sched_order = list(SCHEDULES.keys())
fig = go.Figure(); metrics = [("coverage_at_risk", "coverage-at-risk (orders, upper bound)"),
                              ("forgone_reward_usd", "forgone reward (USD)")]
spans = []
for mi, (col, label) in enumerate(metrics):
    start = len(fig.data)
    for ch in CHAIN_ORDER:
        s = sit[sit.chain == ch].set_index("schedule").reindex(sched_order)
        fig.add_bar(x=sched_order, y=s[col], name=ch, marker_color=COLOR[ch], visible=(mi == 0))
    spans.append((start, len(fig.data), label))
btn = [dict(label=label, method="update",
            args=[{"visible": [a <= t < b for t in range(len(fig.data))]}, {"yaxis": {"title": label}}])
       for a, b, label in spans]
fig.update_layout(barmode="group", title="Sit-out cost vs backoff schedule (per chain)",
                  updatemenus=[dict(buttons=btn, x=1.02, y=1.0)], yaxis_title=metrics[0][1])
fig.show()

## 10. Why is polygon worse: gross/net P&L and the revert-rate artifact

Felix H asked two things: is polygon genuinely less profitable, and is its high revert rate
real. This section adds a GROSS execution-only P&L (slippage minus execution cost, settled rows,
excludes reward/penalty so it is not tautological against the cap), splits P&L settled vs revert,
traces polygon's revert rate to a June-3 services fix, and runs Archit's revert counterfactual.

- gross P&L isolates execution quality; net `pnl_usd` adds the mechanism (reward/penalty).
- distributions use medians/quantiles and clipped ECDFs, never means: the tails are heavy.
- the June-3 cut tests whether polygon's revert rate is partly a phantom-revert data artifact.

In [ ]:
# GROSS execution-only P&L: slippage - execution cost, settled rows only (NaN on reverts).
# Excludes reward/penalty, so it is NOT tautological against the penalty/cap. Uses the raw
# slippage_usd column; execution cost has no USD column in the extract, so derive it from tok.
df["execution_cost_usd"] = df.execution_cost_tok * df.native_usd
df["gross_pnl_usd"] = np.where(df.settled,
                               df.slippage_usd - df.execution_cost_usd, np.nan)

# Per chain: gross (settled-only) and net (pnl_usd), split settled vs revert. Medians only.
g1 = []
for ch in CHAIN_ORDER:
    g = df[df.chain==ch]; s = g[g.settled]; r = g[g.is_revert]
    g1.append(dict(chain=ch, n_settled=len(s), n_revert=len(r),
                   gross_med_settled=round(s.gross_pnl_usd.median(),4),
                   net_med_settled=round(s.pnl_usd.median(),4),
                   net_med_revert=round(r.pnl_usd.median(),4),
                   net_med_all=round(g.pnl_usd.median(),4)))
print("Gross is execution-only (settled); net adds reward/penalty. USD, medians.")
display(pd.DataFrame(g1))

In [ ]:
# P&L distributions per chain, settled vs revert, on a clipped symmetric scale (heavy tails).
# ECDF overlay; gross (settled) on the left panel idea is folded into a dropdown over 3 series.
panels = [("gross_pnl_usd", df.settled,  "gross P&L, settled (USD)"),
          ("pnl_usd",       df.settled,  "net P&L, settled (USD)"),
          ("pnl_usd",       df.is_revert, "net P&L, revert (USD)")]
fig = go.Figure(); buttons = []; spans = {}
for pi,(col,mask,label) in enumerate(panels):
    start = len(fig.data); sub = df[mask]
    for ch in CHAIN_ORDER:
        v = pd.to_numeric(sub.loc[sub.chain==ch, col], errors="coerce").dropna()
        # clip to robust symmetric range so the heavy tails do not flatten the curve
        if len(v):
            lim = np.nanpercentile(np.abs(v), 99)
            v = v.clip(-lim, lim)
        v = np.sort(v.values)
        if len(v)==0:
            fig.add_scatter(x=[], y=[], name=ch, visible=(pi==0)); continue
        y = np.arange(1,len(v)+1)/len(v)
        if len(v)>3000:
            idx = np.linspace(0,len(v)-1,3000).astype(int); v=v[idx]; y=y[idx]
        fig.add_scatter(x=v,y=y,mode="lines",name=ch,line=dict(color=COLOR[ch]),visible=(pi==0))
    spans[pi]=(start,len(fig.data))
n=len(fig.data)
for pi,(col,mask,label) in enumerate(panels):
    vis=[False]*n
    for t in range(*spans[pi]): vis[t]=True
    buttons.append(dict(label=label,method="update",
                        args=[{"visible":vis},{"xaxis":{"title":label}}]))
fig.update_layout(title="P&L distribution by chain (ECDF, clipped at robust 99th pct of |x|)",
                  updatemenus=[dict(buttons=buttons,x=1.02,y=1.0)],
                  yaxis_title="ECDF", xaxis_title=panels[0][2])
fig.add_vline(x=0.0, line_dash="dash", line_color="gray")
fig.show()

# Quantile table (no means): p10/p25/p50/p75/p90 of gross and net per chain.
qs = [.10,.25,.50,.75,.90]
qrows = []
for ch in CHAIN_ORDER:
    s = df[(df.chain==ch)&df.settled]
    for nm,series in [("gross(settled)", s.gross_pnl_usd),
                      ("net(settled)",   s.pnl_usd),
                      ("net(revert)",    df[(df.chain==ch)&df.is_revert].pnl_usd)]:
        q = series.dropna().quantile(qs)
        qrows.append(dict(chain=ch, series=nm, **{f"p{int(p*100)}":round(q.loc[p],4) for p in qs}))
display(pd.DataFrame(qrows))

In [ ]:
# Polygon revert-rate drivers.
# (a) by USD size band, polygon vs bnb (size is the obvious confounder; check it is not enough)
bands2 = [0,100,1_000,10_000,100_000,np.inf]
labels2 = ["<$100","$100-1k","$1k-10k","$10k-100k",">$100k"]
df["size_band_pl"] = pd.cut(df.size_usd, bands2, labels=labels2)
sb = []
for ch in ["polygon","bnb"]:
    for b in labels2:
        g = df[(df.chain==ch)&(df.size_band_pl==b)]
        if len(g)<30: continue
        p,lo,hi = wilson(int(g.is_revert.sum()), len(g))
        sb.append(dict(chain=ch, band=b, n=len(g), revert_rate=round(p,4)))
sb = pd.DataFrame(sb)
print("median size_usd:", df.groupby("chain").size_usd.median().round(1).to_dict())
print("revert rate by size band (polygon stays ~5x bnb within every band -> size is not the driver):")
display(sb.pivot(index="band", columns="chain", values="revert_rate").reindex(labels2))

# (b) by solver and by top token-pair, polygon
poly = df[df.chain=="polygon"].copy()
by_solver = (poly.groupby("solver_name")
             .agg(n=("is_revert","size"), revert_rate=("is_revert","mean"))
             .query("n>=200").sort_values("revert_rate",ascending=False).round(3))
print("polygon revert rate by solver (n>=200):"); display(by_solver.head(10))
poly["pair"] = poly.sell_token.astype(str).str[:10]+"/"+poly.buy_token.astype(str).str[:10]
by_pair = (poly.groupby("pair")
           .agg(n=("is_revert","size"), revert_rate=("is_revert","mean"))
           .query("n>=100").sort_values("revert_rate",ascending=False).round(3))
print("polygon revert rate by top token-pair (n>=100):"); display(by_pair.head(10))

In [ ]:
# (c) CRUCIAL: polygon revert rate PRE vs POST the June-3 services fix.
# cowprotocol/services PR #4465 (merged 2026-06-03) fixed FALSE-POSITIVE reverts from conflicting
# pending-block simulations in PUBLIC mempools. Polygon is a public-mempool chain, so pre-June-3
# reverts may be inflated by phantom reverts (the settlement actually landed). bnb is a control.
df["auction_date"] = pd.to_datetime(df.auction_timestamp, errors="coerce").dt.date
CUTOFF = pd.Timestamp("2026-06-03").date()
df["fix_period"] = np.where(df.auction_date < CUTOFF, "pre-Jun3", "post-Jun3")
pp = []
for ch in CHAIN_ORDER:
    g = df[df.chain==ch]
    for per in ["pre-Jun3","post-Jun3"]:
        gp = g[g.fix_period==per]
        if len(gp)==0: continue
        p,lo,hi = wilson(int(gp.is_revert.sum()), len(gp))
        pp.append(dict(chain=ch, period=per, n=len(gp),
                       revert_rate=round(p,4), lo=round(lo,4), hi=round(hi,4)))
pp = pd.DataFrame(pp)
display(pp)

poly_pre  = pp[(pp.chain=="polygon")&(pp.period=="pre-Jun3")].revert_rate.iloc[0]
poly_post = pp[(pp.chain=="polygon")&(pp.period=="post-Jun3")].revert_rate.iloc[0]
print(f"polygon revert rate: {poly_pre:.1%} pre-Jun3  ->  {poly_post:.1%} post-Jun3 "
      f"(drop of {poly_pre-poly_post:.1%}pts).")
print("bnb (public-mempool control) barely moves, so the polygon drop is the fix removing phantom")
print("reverts, not a change in solver behavior. Polygon's headline ~23% revert rate is partly a")
print("data artifact: the post-fix rate is the honest number to use against the cap.")

fig = go.Figure()
for per,dash in [("pre-Jun3","solid"),("post-Jun3","dot")]:
    d = pp[pp.period==per]
    fig.add_bar(x=d.chain, y=d.revert_rate, name=per,
                error_y=dict(type="data", symmetric=False,
                             array=d.hi-d.revert_rate, arrayminus=d.revert_rate-d.lo))
fig.update_layout(title="Revert rate pre vs post Jun-3 services fix (#4465), Wilson 95% CI",
                  barmode="group", yaxis_tickformat=".0%")
fig.show()

In [ ]:
# (c2) Avalanche: a SECOND driver fix on a different chain/date. cowprotocol/services PR #4510
# (merged 2026-06-11) fixed false-positive settlement cancellations from node receipt lag, and
# called out Avalanche specifically as the main source of its "too many failing settlements" alert.
# So cut avalanche's revert rate pre vs post Jun-11 (bnb as a control), the same artifact test.
CUT2 = pd.Timestamp("2026-06-11").date()
df["fix_period2"] = np.where(df.auction_date < CUT2, "pre-Jun11", "post-Jun11")
av = []
for ch in [c for c in ["avalanche", "bnb"] if c in CHAIN_ORDER]:
    g = df[df.chain == ch]
    for per in ["pre-Jun11", "post-Jun11"]:
        gp = g[g.fix_period2 == per]
        if len(gp) == 0: continue
        p, lo, hi = wilson(int(gp.is_revert.sum()), len(gp))
        av.append(dict(chain=ch, period=per, n=len(gp),
                       revert_rate=round(p, 4), lo=round(lo, 4), hi=round(hi, 4)))
av = pd.DataFrame(av); display(av)

if not av.empty and (av.chain == "avalanche").any():
    a_pre = av[(av.chain == "avalanche") & (av.period == "pre-Jun11")].revert_rate
    a_post = av[(av.chain == "avalanche") & (av.period == "post-Jun11")].revert_rate
    if len(a_pre) and len(a_post):
        print(f"avalanche revert rate: {a_pre.iloc[0]:.1%} pre-Jun11  ->  {a_post.iloc[0]:.1%} post-Jun11.")
        print("PR #4510 targeted Avalanche node-receipt-lag false positives; a drop here is the same")
        print("data-artifact story as polygon, on a different chain and a later fix date. Post-Jun11 is")
        print("a short window (the data ends Jun-22), so the CI is wider.")

    fig = go.Figure()
    for per in ["pre-Jun11", "post-Jun11"]:
        d = av[av.period == per]
        fig.add_bar(x=d.chain, y=d.revert_rate, name=per,
                    error_y=dict(type="data", symmetric=False,
                                 array=d.hi - d.revert_rate, arrayminus=d.revert_rate - d.lo))
    fig.update_layout(title="Avalanche revert rate pre vs post Jun-11 services fix (#4510), Wilson 95% CI",
                      barmode="group", yaxis_tickformat=".0%")
    fig.show()

In [ ]:
# Polygon counterfactual (Archit's idea): hold polygon's OWN settled and revert P&L
# distributions, swap only its revert rate for bnb's. How much of the gap is the revert rate?
poly = df[df.chain=="polygon"]; bnb = df[df.chain=="bnb"]
p_settle_poly = poly.settled.mean(); p_settle_bnb = bnb.settled.mean()
med_poly_settled = poly.loc[poly.settled,"pnl_usd"].median()
med_poly_revert  = poly.loc[poly.is_revert,"pnl_usd"].median()
med_bnb_settled  = bnb.loc[bnb.settled,"pnl_usd"].median()
med_bnb_revert   = bnb.loc[bnb.is_revert,"pnl_usd"].median()

exp_poly_own    = p_settle_poly*med_poly_settled + (1-p_settle_poly)*med_poly_revert
exp_poly_bnbrr  = p_settle_bnb *med_poly_settled + (1-p_settle_bnb )*med_poly_revert
exp_bnb_own     = p_settle_bnb *med_bnb_settled  + (1-p_settle_bnb )*med_bnb_revert

cf = pd.DataFrame([
    dict(scenario="polygon, own revert rate",      p_settle=round(p_settle_poly,4), exp_pnl_usd=round(exp_poly_own,4)),
    dict(scenario="polygon, bnb's revert rate",    p_settle=round(p_settle_bnb,4),  exp_pnl_usd=round(exp_poly_bnbrr,4)),
    dict(scenario="bnb, own (reference)",          p_settle=round(p_settle_bnb,4),  exp_pnl_usd=round(exp_bnb_own,4)),
])
display(cf)
gap = exp_bnb_own - exp_poly_own
closed = exp_poly_bnbrr - exp_poly_own
print(f"polygon-vs-bnb expected-P&L gap = {gap:.4f} USD/attempt.")
print(f"Lowering polygon's revert rate to bnb's closes {closed:.4f} USD = "
      f"{closed/gap:.0%} of the gap. Most of polygon's P&L disadvantage is the revert rate,")
print("not worse settled execution. Combined with the Jun-3 result, much of THAT revert rate is")
print("itself a pre-fix artifact, so the cap is unlikely to be the main driver.")

### Caveat: consistency-reward redistribution on polygon

`pnl_usd` charges the full penalty as a solver loss. On chains where penalties feed the consistency
budget and are redistributed back to solvers (polygon), the charged penalty overstates the realized
loss: some of it returns to solvers. So polygon's negative net P&L is biased downward (too negative)
and the polygon-vs-bnb P&L gap is overstated.

- gross execution P&L is unaffected (it excludes reward/penalty), so the gross finding stands.
- we cannot net out the rebate here: the redistribution flows are not in this extract.
- data ask: per-solver consistency-budget inflows on polygon, to compute realized (post-rebate) P&L.

## 11. Cap quality: position on the uncapped-penalty distribution, not a fitted number

Felix H's observation: plot, per chain, the distribution of the UNCAPPED penalty relative to the cap and
the curves look similar up to a horizontal SHIFT, and that shift is a different choice of cap. The aim
here is to turn "where the cap sits on its own chain's distribution" into the quality measure, and to
pick the cap from inputs we can observe at launch.

Design constraint (Felix H): a first-principles cap must depend only on inputs that are observable
on-chain or self-referential (derivable from the current accounting week's own data). It must NOT be
fitted to a historical target, because a brand-new chain has no history to fit to.

Archit's caveat: the curves will not lie exactly on top of each other. Chains differ in liquidity depth
and token-pair mix, so after the best shift there is residual spread. We measure that spread rather than
assume it away.

How to read this section:
- We work on reverts with a positive uncapped penalty and a positive cap (the population the cap acts on).
- Uncapped penalty in USD is the most ethereum-outlier-exposed field, so we winsorize per chain at p99.5
  before any quantile, sum, or alignment. Raw tails are printed beside so the gating is visible.

In [ ]:
# Build the working set: reverts where the cap can bind, with a USD uncapped penalty.
df["penalty_uncapped_usd"] = df.penalty_uncapped_tok * df.native_usd

def winsor_per_chain(frame, col, hi=0.995):
    out = pd.Series(np.nan, index=frame.index)
    for ch in CHAIN_ORDER:
        m = frame.chain == ch
        s = frame.loc[m, col]
        q = s[s > 0].quantile(hi)
        out.loc[m] = s.clip(upper=q)
    return out

capq = df[df.is_revert & (df.penalty_uncapped_usd > 0) & (df.penalty_cap_usd > 0)].copy()
capq["unc_w"] = winsor_per_chain(capq, "penalty_uncapped_usd", hi=0.995)

# Show the gating worked: raw vs winsorized tail (note ethereum max collapses).
tail = []
for ch in CHAIN_ORDER:
    g = capq[capq.chain == ch]
    tail.append(dict(chain=ch, reverts=len(g),
        raw_p99=round(g.penalty_uncapped_usd.quantile(.99),2),
        raw_p999=round(g.penalty_uncapped_usd.quantile(.999),2),
        raw_max=g.penalty_uncapped_usd.max(),
        winsor_max=round(g.unc_w.max(),2)))
print("uncapped-penalty USD tails, raw vs winsorized (per-chain p99.5 clip):")
display(pd.DataFrame(tail))

### 11a. The uncapped-penalty distribution and the cap's position on it

ECDF of the uncapped penalty (USD, winsorized, log x), NOT divided by the cap. The dashed vertical line
is each chain's current cap. Where the line crosses a chain's curve is the fraction of reverts the cap
leaves UNCAPPED. One minus that is the capped fraction. If Felix H's claim holds, the three curves have a
common shape and the caps sit at very different heights on them: that height is the quality signal.

In [ ]:
# ECDF (log x) of winsorized uncapped penalty, with each chain's cap as a vertical line.
fig = go.Figure()
for ch in CHAIN_ORDER:
    v = np.sort(capq.loc[capq.chain == ch, "unc_w"].values)
    v = v[v > 0]
    y = np.arange(1, len(v)+1) / len(v)
    if len(v) > 3000:
        idx = np.linspace(0, len(v)-1, 3000).astype(int); v = v[idx]; y = y[idx]
    fig.add_scatter(x=v, y=y, mode="lines", name=ch, line=dict(color=COLOR[ch]))
    capval = capq.loc[capq.chain == ch, "penalty_cap_usd"].median()
    fig.add_vline(x=capval, line_dash="dash", line_color=COLOR[ch])
fig.update_layout(title="Uncapped penalty (USD, winsorized) by chain; dashed = current cap",
                  yaxis_title="ECDF", xaxis_title="uncapped penalty (USD)")
fig.update_xaxes(type="log")
fig.show()

# Capped fraction = share of reverts where cap binds = quantile position of cap on the uncapped dist.
pos = []
for ch in CHAIN_ORDER:
    g = capq[capq.chain == ch]
    capval = g.penalty_cap_usd.median()
    cap_position = (g.unc_w <= capval).mean()      # percentile the cap sits at
    pos.append(dict(chain=ch, cap_usd=round(capval,3),
                    median_uncapped_usd=round(g.unc_w.median(),4),
                    cap_over_median=round(capval / g.unc_w.median(), 3),
                    cap_percentile=round(cap_position,3),
                    capped_fraction=round(1 - cap_position,3),
                    capped_fraction_all_reverts=round(
                        df.loc[df.is_revert & (df.chain==ch), "cap_binds"].mean(), 3)))
capq_pos = pd.DataFrame(pos)
print("Cross-check vs sample target: bnb ~0.075, polygon ~0.40, eth ~0.44 (all-reverts column).")
display(capq_pos)

### 11b. Quantify the shift, and read it as a relative cap

Estimate, per chain, the multiplicative factor that best aligns each uncapped-penalty distribution onto
a reference chain (polygon). We do it in log space, robustly: the optimal shift is the mean gap between
the chains' log-quantile curves over a robust quantile band, and the best multiplicative factor is its
exponential. The residual spread after the shift is Archit's caveat made numeric: zero would mean the
curves are identical up to scale; a positive residual std is the liquidity / token-mix difference.

The alignment factor is itself a relative-cap statement. If chain X's uncapped distribution is k times
chain Y's, then a cap that occupies the same position on chain Y should be k times larger on chain X.

In [ ]:
ref = "polygon"
qs = np.linspace(0.10, 0.90, 17)        # robust quantile band, avoids the winsorized tail
logq = {ch: np.log(np.quantile(capq.loc[capq.chain==ch, "unc_w"].values, qs)) for ch in CHAIN_ORDER}

align = []
for ch in CHAIN_ORDER:
    shift = float(np.mean(logq[ref] - logq[ch]))           # least-squares shift in log space
    factor = float(np.exp(shift))                          # multiplicative factor onto reference
    resid = logq[ref] - (logq[ch] + shift)                 # how well curves overlay after the shift
    align.append(dict(chain=ch,
                      align_factor_to_polygon=round(factor,4),
                      resid_std_log=round(float(resid.std()),4),
                      resid_range_log=round(float(resid.max()-resid.min()),4)))
align = pd.DataFrame(align)
print(f"Alignment onto {ref}: factor is the multiplicative shift; resid_* is leftover spread (Archit).")
display(align)

# Visual: overlay the shifted curves to show they nearly coincide after the shift.
fig = go.Figure()
for ch in CHAIN_ORDER:
    factor = float(align.loc[align.chain==ch, "align_factor_to_polygon"].iloc[0])
    v = np.sort((capq.loc[capq.chain==ch, "unc_w"] * factor).values)   # shifted onto polygon
    v = v[v > 0]; y = np.arange(1, len(v)+1)/len(v)
    if len(v) > 3000:
        idx = np.linspace(0, len(v)-1, 3000).astype(int); v = v[idx]; y = y[idx]
    fig.add_scatter(x=v, y=y, mode="lines", name=f"{ch} x{factor:.3g}", line=dict(color=COLOR[ch]))
fig.update_layout(title="Uncapped-penalty ECDFs after the best multiplicative shift onto polygon",
                  yaxis_title="ECDF", xaxis_title="shifted uncapped penalty (USD)")
fig.update_xaxes(type="log")
fig.show()

### 11c. A self-referential cap rule and what it does per chain

The cap-as-position view gives a rule that needs only the current week's own uncapped-penalty
distribution, which is observable the moment a chain has any reverts:

    cap = q-th percentile of the current week's uncapped-penalty distribution.

By construction this puts the cap at the same percentile on every chain, so it makes the chains
comparable by the only thing that is policy-relevant: the fraction of reverts that get clipped
(= 1 - q). A median-multiple form, cap = k x median(uncapped), is the same idea expressed through a
scale statistic; it is more robust on thin new chains but only holds the capped fraction fixed if the
distribution shape is stable (Archit's caveat again, since shape is exactly what differs).

Below: what cap and capped fraction the percentile rule yields per chain, and the multiple k that the
CURRENT caps implicitly choose. The current caps are NOT at a consistent position: polygon and ethereum
sit near their median (k ~ 1.1, ~48-49% capped) while bnb's cap sits far out in the tail (k ~ 42, ~9%
capped). That inconsistency is the thing the rule fixes.

In [ ]:
# Rule A: cap = p_q of the current-week uncapped distribution (winsorized). Target capped frac = 1 - q.
ruleA = []
for q in [0.75, 0.90, 0.95]:
    for ch in CHAIN_ORDER:
        g = capq[capq.chain == ch]
        cap_rule = float(np.quantile(g.unc_w.values, q))
        induced = float((g.penalty_uncapped_usd > cap_rule).mean())   # check on raw (pre-winsor) tail
        ruleA.append(dict(rule=f"p{int(q*100)} (target capped {1-q:.2f})", chain=ch,
                          cap_usd=round(cap_rule,3), induced_capped_frac=round(induced,3),
                          current_cap_usd=round(g.penalty_cap_usd.median(),3),
                          current_capped_frac=round(g.cap_binds.mean(),3)))
print("Rule A: cap = percentile of the current-week uncapped-penalty distribution")
display(pd.DataFrame(ruleA))

# Rule B framing: the multiple k = current_cap / median_uncapped that each chain implicitly uses today.
implied = []
for ch in CHAIN_ORDER:
    g = capq[capq.chain == ch]
    med = g.unc_w.median()
    implied.append(dict(chain=ch, median_uncapped_usd=round(med,4),
                        current_cap_usd=round(g.penalty_cap_usd.median(),3),
                        implied_k=round(g.penalty_cap_usd.median()/med,2),
                        current_capped_frac=round(g.cap_binds.mean(),3)))
print("Implied multiple k = current cap / median uncapped (today's caps are at inconsistent positions):")
display(pd.DataFrame(implied))

### 11d. Design takeaway

- Treat the cap as a POSITION on the chain's own uncapped-penalty distribution, not as a fitted dollar
  number. The natural quality measure of a cap is the capped fraction it produces (equivalently, the
  percentile the cap sits at). On this sample the current caps are at very inconsistent positions:
  polygon and ethereum clip about half of reverting penalties, bnb under a tenth.
- The uncapped-penalty distributions are similar up to a multiplicative shift (Felix H). The best shift
  onto polygon is roughly x3.4 for bnb and x0.10 for ethereum, with modest residual spread after the
  shift (Archit: liquidity and token-mix differences, not nothing, but small relative to the shift).
  That shift IS a relative-cap choice: same position, scaled by the factor.
- A self-referential rule, cap = q-th percentile of the current accounting week's own uncapped-penalty
  distribution, satisfies the launch constraint: it needs only the current week's observable data, never
  a historical fit, so it can be set on a brand-new chain as soon as it has reverts. It makes chains
  comparable by holding the capped fraction fixed at 1 - q.
- Pick q (equivalently the capped fraction) from policy, not from the data: it is the one free parameter
  and it is the same number on every chain. A median-multiple form (k x median) is the robust small-sample
  approximation; prefer the direct percentile once a week has enough reverts.

## 12. Daily success rate and outlier-day correction

- per-chain daily success rate (1 - revert rate) over time, with the consistency-rewards activation
  (2026-04-07) and the two driver false-positive fixes (#4465 on 2026-06-03, #4510 on 2026-06-11)
  marked. Toggle chains in the legend.
- polygon and others have low-success outlier days (driver false positives plus the consistency-
  rewards onset, with normal weeks in between), so a single pre/post split is misleading. We flag
  outlier days and re-report revert rate and cap-binding with them removed.

In [ ]:
# Daily success (settled) rate per chain, Wilson CI, thin days dropped.
MIN_N = 30
dd = df.copy()
dd["day"] = pd.to_datetime(dd.auction_timestamp, errors="coerce").dt.floor("D")
dd = dd[dd.day.notna()]
daily = (dd.groupby(["chain","day"])
         .agg(n=("is_revert","size"), succ=("is_revert", lambda s: int((~s).sum()))).reset_index())
daily = daily[daily.n >= MIN_N].copy()
ci = daily.apply(lambda r: wilson(r.succ, r.n), axis=1, result_type="expand")
daily[["success_rate","lo","hi"]] = ci

MARKS = [("2026-04-07","consistency rewards"), ("2026-06-03","driver fix #4465"),
         ("2026-06-11","driver fix #4510")]
fig = go.Figure()
for ch in CHAIN_ORDER:
    g = daily[daily.chain==ch].sort_values("day")
    fig.add_scatter(x=g.day, y=g.success_rate, mode="lines+markers", name=ch,
                    line=dict(color=COLOR[ch]), marker=dict(size=3))
for ds,lab in MARKS:
    ts = pd.Timestamp(ds)
    fig.add_shape(type="line", xref="x", yref="paper", x0=ts, x1=ts, y0=0, y1=1,
                  line=dict(color="gray", dash="dash"))
    fig.add_annotation(x=ts, y=1.0, yref="paper", text=lab, showarrow=False,
                       textangle=-90, font=dict(size=9), xshift=-7)
fig.update_layout(title="Daily success (settled) rate by chain (toggle chains in legend)",
                  yaxis_title="success rate", yaxis_tickformat=".0%", yaxis_range=[0,1])
fig.show()

In [ ]:
# Flag low-success outlier days per chain (success_rate well below the chain median), then
# re-report revert rate and cap-binding with those days removed.
med = daily.groupby("chain").success_rate.transform("median")
daily["is_outlier_day"] = daily.success_rate < (med - 0.10)
flag = (daily.groupby("chain")
        .agg(days=("day","size"), outlier_days=("is_outlier_day","sum"),
             median_success=("success_rate","median")).reindex(CHAIN_ORDER))
flag["outlier_share"] = (flag.outlier_days / flag.days).round(3)
print("low-success outlier days per chain (daily success_rate < chain median - 10pts):")
display(flag.round(3))

# map attempts to outlier (chain, day) pairs and recompute clean stats
outlier_pairs = set(map(tuple, daily.loc[daily.is_outlier_day, ["chain","day"]].itertuples(index=False, name=None)))
dd["is_outlier_day"] = [(c,d) in outlier_pairs for c,d in zip(dd.chain, dd.day)]
clean = dd[~dd.is_outlier_day]
cmp_rows = []
for ch in CHAIN_ORDER:
    a = dd[dd.chain==ch]; b = clean[clean.chain==ch]
    if len(a)==0: continue
    cmp_rows.append(dict(chain=ch,
        revert_raw=round(a.is_revert.mean(),4),
        revert_clean=round(b.is_revert.mean(),4) if len(b) else np.nan,
        cap_binds_raw=round(a.loc[a.is_revert,"cap_binds"].mean(),4),
        cap_binds_clean=round(b.loc[b.is_revert,"cap_binds"].mean(),4) if len(b) and b.is_revert.any() else np.nan,
        attempts_dropped=len(a)-len(b)))
print("revert rate and cap-binding: raw vs outlier-days-removed (the artifact correction):")
display(pd.DataFrame(cmp_rows))